# 100m Grid → 자치구(Gu) 매핑

## 목적
- 서울 100m grid(cell_id)를 자치구 경계와 공간적으로 매핑
- 각 cell_id가 속한 자치구 코드/이름 부여
- 이후 모든 자치구 단위 통계의 기준 테이블 생성

## 입력 데이터
- 100m Grid (cell_id 기준)
- 서울 자치구 경계 SHP (원본)

## 출력 데이터
- grid_to_gu.csv
  - cell_id
  - gu_code
  - gu_name

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [2]:
# =========================
# Global Configuration
# =========================

# Grid 데이터 경로 (geometry 포함)
GRID_GEOJSON_PATH = "../data_processed/processed/grid/seoul_grid_100m.geojson"

# 자치구 SHP 경로 (raw, 절대 수정 금지)
GU_SHP_PATH = "../data_processed/raw/admin_boundary/seoul_gu/BND_SIGUNGU_PG.shp"

# 출력 경로
OUTPUT_PATH = "../data_processed/processed/admin_mapping/grid_to_gu.csv"

# 좌표계
TARGET_CRS = "EPSG:3857"

In [3]:
# =========================
# Load 100m Grid
# =========================

grid_gdf = gpd.read_file(GRID_GEOJSON_PATH)

print("Grid rows:", len(grid_gdf))
print("Grid CRS:", grid_gdf.crs)

grid_gdf.head()

Grid rows: 92398
Grid CRS: EPSG:4326


,id,count,cell_id,geometry
0,+141141+45163,1,1411415045163,"POLYGON ((126.78912 37.55178, 126.79002 37.551..."
1,+141141+45164,1,1411415045164,"POLYGON ((126.78912 37.5525, 126.79002 37.5525..."
2,+141141+45165,1,1411415045165,"POLYGON ((126.78912 37.55321, 126.79002 37.553..."
3,+141142+45158,1,1411425045158,"POLYGON ((126.79002 37.54822, 126.79091 37.548..."
4,+141142+45159,1,1411425045159,"POLYGON ((126.79002 37.54894, 126.79091 37.548..."


In [4]:
# =========================
# Convert Grid CRS to Meter-based
# =========================

grid_gdf = grid_gdf.to_crs(TARGET_CRS)

print("Converted Grid CRS:", grid_gdf.crs)

Converted Grid CRS: EPSG:3857


In [5]:
# =========================
# Load GU Boundary SHP
# =========================

gu_gdf = gpd.read_file(GU_SHP_PATH)

print("GU rows:", len(gu_gdf))
gu_gdf.head()

GU rows: 252


,BASE_DATE,SIGUNGU_CD,SIGUNGU_NM,geometry
0,20240630,11060,동대문구,"POLYGON ((206125.534 556523.929, 206126.012 55..."
1,20240630,11070,중랑구,"POLYGON ((209826.336 557905.317, 209881.53 557..."
2,20240630,11080,성북구,"POLYGON ((198584.191 559647.276, 198605.314 55..."
3,20240630,11090,강북구,"POLYGON ((200404.977 565045.309, 200487.736 56..."
4,20240630,11100,도봉구,"POLYGON ((201748.532 566815.017, 201953.244 56..."


In [6]:
# 서울 지역만 필터링
gu_seoul = gu_gdf[
    gu_gdf["SIGUNGU_CD"].astype(str).str.startswith("11")
]

print(len(gu_seoul))

25


In [7]:
gu_seoul = gu_seoul.to_crs(TARGET_CRS)

print("Converted GU CRS:", gu_seoul.crs)

Converted GU CRS: EPSG:3857


In [8]:
# =========================
# centroid 기반 join + nearest fallback
# =========================

grid_centroid = grid_gdf.copy()
grid_centroid["geometry"] = grid_centroid.geometry.centroid

joined = gpd.sjoin(
    grid_centroid,
    gu_seoul,
    how="left",
    predicate="within"
)

null_mask = joined["SIGUNGU_CD"].isna()

if null_mask.sum() > 0:

    fallback = gpd.sjoin_nearest(
        grid_centroid[null_mask],
        gu_seoul,
        how="left",
        distance_col="dist"
    )

    joined.loc[null_mask, ["SIGUNGU_CD", "SIGUNGU_NM"]] = \
        fallback[["SIGUNGU_CD", "SIGUNGU_NM"]].values


print("Join completed")
joined.head()

Join completed


,id,count,cell_id,geometry,index_right,BASE_DATE,SIGUNGU_CD,SIGUNGU_NM
0,+141141+45163,1,1411415045163,POINT (14114150 4516350),10.0,20240630,11160,강서구
1,+141141+45164,1,1411415045164,POINT (14114150 4516450),10.0,20240630,11160,강서구
2,+141141+45165,1,1411415045165,POINT (14114150 4516550),10.0,20240630,11160,강서구
3,+141142+45158,1,1411425045158,POINT (14114250 4515850),10.0,20240630,11160,강서구
4,+141142+45159,1,1411425045159,POINT (14114250 4515950),10.0,20240630,11160,강서구


In [9]:
grid_to_gu = joined[
    ["cell_id", "SIGUNGU_CD", "SIGUNGU_NM"]
].copy()

print("Null:", grid_to_gu["SIGUNGU_CD"].isna().sum())
print("Duplicate:", grid_to_gu["cell_id"].duplicated().sum())


Null: 0
Duplicate: 0


In [10]:
grid_to_gu = joined[
    ["cell_id", "SIGUNGU_CD", "SIGUNGU_NM"]
].copy()

print("Null GU:", grid_to_gu["SIGUNGU_CD"].isna().sum())
print("Duplicate cell:", grid_to_gu["cell_id"].duplicated().sum())
print("Total rows:", len(grid_to_gu))

Null GU: 0
Duplicate cell: 0
Total rows: 92398


In [12]:
grid_to_gu.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", OUTPUT_PATH)

Saved: ../data_processed/processed/admin_mapping/grid_to_gu.csv
